In [0]:
from pyspark.sql.functions import *

from pyspark.sql.types import *

source_path="/Volumes/retail_lakehouse/bronze/raw_files/orders/"

schema_location = "/Volumes/retail_lakehouse/bronze/raw_files/schema_tracking/orders/"

orders_stream_df = spark.readStream.format("cloudfiles").option("cloudfiles.format","csv").option("header", "true").option("cloudfiles.schemaLocation", schema_location).load(source_path).withColumn("Injection_timestamp", current_timestamp()).withColumn("file_path", col("_metadata.file_path")).drop(col("_rescued_data"))

checkpoint_path ="/Volumes/retail_lakehouse/bronze/raw_files/checkpoints/orders/"
bronze_table = "retail_lakehouse.bronze.orders_bronze"

bronze_stream = orders_stream_df.writeStream.format("delta").option("checkpointLocation", checkpoint_path).trigger(availableNow=True).toTable(bronze_table)